# multi_1h_jma_atr_038

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (30).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 15 |
| Detected functions | resample_1hr_custom, compute_jma_indicator, cy_backtest_trades, backtest_trades, calculate_metrics_fast, main |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Copy of Binance_Crypto_T7_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2024-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['XRPUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

def compute_jma_indicator(
    df,
    smooth_length=14,
    normalization_period=60,
    upper_threshold=1.75,
    lower_threshold=-1.25,
    rs_period=14,
    rs_percentile=30
):
    # === Jurik Moving Average (JMA) Calculation ===
    price_series = df['close']
    alpha = 2.0 / (smooth_length + 1)

    jma_values = [price_series.iloc[0]]
    for price in price_series.iloc[1:]:
        jma_values.append(jma_values[-1] + (price - jma_values[-1]) * alpha)

    jma_series = pd.Series(jma_values, index=price_series.index)

    # === Normalization ===
    rolling_mean = jma_series.rolling(window=normalization_period).mean()
    rolling_std = jma_series.rolling(window=normalization_period).std()
    normalized_jma = (jma_series - rolling_mean) / rolling_std

    # === Rogers-Satchell Volatility Calculation ===
    open_prices = df['open'].replace(0, np.nan)
    high_prices = df['high']
    low_prices = df['low']
    close_prices = df['close'].replace(0, np.nan)

    log_hc = np.log(high_prices / close_prices)
    log_ho = np.log(high_prices / open_prices)
    log_lc = np.log(low_prices / close_prices)
    log_lo = np.log(low_prices / open_prices)

    rs_volatility = (log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()

    # === RS Volatility Filter ===
    rs_threshold = rs_volatility.rolling(window=normalization_period).quantile(rs_percentile / 100.0)

    # === Signal Generation ===
    long_signals = (normalized_jma > upper_threshold) & (rs_volatility > rs_threshold)
    short_signals = (normalized_jma < lower_threshold) & (rs_volatility > rs_threshold)

    signals = pd.Series(0, index=df.index)
    signals[long_signals] = 1
    signals[short_signals] = 2

    # === Attach Results to DataFrame ===
    df['JMA'] = jma_series
    df['Normalized_JMA'] = normalized_jma
    df['RS_Volatility'] = rs_volatility
    df['RS_Threshold'] = rs_threshold
    df['Signal'] = signals

    return df

# Assumes `data` is already loaded with: Ticker, Date, open, high, low, close, Signal
data['Date'] = pd.to_datetime(data['Date'])

# Apply your signal logic
data = data.groupby('Ticker', group_keys=False).apply(compute_jma_indicator)

# Shift signals for next-bar execution
data['Signal'] = data.groupby('Ticker')['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(np.int8)
data

In [ ]:
summary = data['Signal'].value_counts().reset_index()
summary.columns = ['Signal', 'Count']
print("Number of unique exit reasons:", len(summary))
print(summary)

In [ ]:
load_ext cython

In [ ]:
%%cython
import numpy as np
cimport numpy as np

def cy_backtest_trades(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double tp_pct,
    double sl_pct,
    double entry_sl_pct
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        bint exit_next_open = False

        # Trade State
        double entry_price = 0.0, quantity = 0.0
        object entry_time = None
        int direction = 0  # 1 = long, -1 = short
        double tp_price = 0.0, sl_price = 0.0, esl_price = 0.0
        object last_exit_time = None

        # Output
        list trades = []
        double cumulative_pnl = 0.0

        # Candle data
        object time
        double open_, high, low, close
        int signal

        # Exit
        double exit_price, pnl, fee, net_pnl
        str reason

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        # === Exit on Open (delayed SL confirmation) ===
        if exit_next_open:
            exit_price = open_ * (1 + slippage_rate) if direction == -1 else open_ * (1 - slippage_rate)
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl

            trades.append((
                ticker, entry_time, time, entry_price, exit_price, net_pnl,
                "SL Close Confirmed", cumulative_pnl,
                "long" if direction == 1 else "short"
            ))

            # Reset
            position_open = False
            exit_next_open = False
            last_exit_time = time
            continue

        # === Entry Logic ===
        if not position_open and time != last_exit_time and signal in (1, 2):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            position_open = True

            # Initialize SL/TP
            sl_price = entry_price * (1 - sl_pct * direction)
            tp_price = entry_price * (1 + tp_pct * direction)
            esl_price = entry_price * (1 - entry_sl_pct * direction)

            # === Entry SL Check ===
            if (direction == 1 and low <= esl_price) or (direction == -1 and high >= esl_price):
                exit_price = esl_price
                pnl = (exit_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price, net_pnl,
                    "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open = False
                last_exit_time = time
                continue

            # === Delayed SL Exit Check ===
            if (direction == 1 and close <= sl_price) or (direction == -1 and close >= sl_price):
                exit_next_open = True

            continue  # Skip to next candle

        # === Position Management ===
        if position_open:
            # --- Trailing SL Update ---
            if direction == 1:
                sl_price = max(sl_price, high * (1 - sl_pct))
            else:
                sl_price = min(sl_price, low * (1 + sl_pct))

            # --- Trailing TP Update ---
            if direction == 1:
                tp_price = max(tp_price, high * (1 - tp_pct))
            else:
                tp_price = min(tp_price, low * (1 + tp_pct))

            # --- Exit Checks ---
            if (direction == 1 and high >= tp_price) or (direction == -1 and low <= tp_price):
                exit_price = tp_price
                reason = "Trailing TP"
            elif (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                reason = "Stop Loss"
            else:
                continue  # Still holding

            # --- Exit Execution ---
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl

            trades.append((
                ticker, entry_time, time, entry_price, exit_price, net_pnl,
                reason, cumulative_pnl,
                "long" if direction == 1 else "short"
            ))

            position_open = False
            last_exit_time = time

    return trades


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def backtest_trades(
    data: pd.DataFrame,
    tp_pct: float,
    sl_pct: float,
    entry_sl_pct: float
) -> pd.DataFrame:

    initial_capital = 100000
    fee_rate = 0.00025  # 0.04% round trip
    slippage_rate = 0.0006  # 0.05% round trip

    # Convert 'Date' to datetime and avoid deprecated warning
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Prepare each ticker's data for cython backtest
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []

    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            tp_pct,
            sl_pct,
            entry_sl_pct  # Used for Entry SL check
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

tradebook = backtest_trades(
    data,
    tp_pct=0.075,            # 6% TP
    sl_pct=0.001,           # 0.1% SL
    entry_sl_pct=0.015,      # 2% Entry SL
)

tradebook.to_csv("/content/drive/MyDrive/tradebook_SOLUSDT_2022_24.csv", index=False)
tradebook


In [ ]:
import os
import time
import pickle
import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm

# === Hyperparameter Grid ===
param_grid = {
    'smooth_length': [14],
    'norm_period': [36, 60, 84],
    'upper_threshold': [1.5, 1.75, 2],
    'lower_threshold': [-1, -1.25, -1.5],
    'rs_period': [3, 10, 14, 21, 28],
    'rs_percentile': [10, 30, 50, 70, 80, 90],
    'tp_pct': [0.05, 0.07],
    'sl_pct': [0.001, 0.002],
    'entry_sl_pct': [0.02, 0.03, 0.04]
}
keys, values = zip(*param_grid.items())
param_combinations = list(product(*values))
print(f"Total combinations: {len(param_combinations)}")

# === Paths ===
csv_path = "/content/drive/MyDrive/hyperparameter_results_XRPUSDT.csv"
cache_path = "/content/drive/MyDrive/hyperparameter_checkpoint_XRPUSDT.pkl"

# === Fast Metrics ===
def calculate_metrics_fast(tradebook):
    if tradebook.empty:
        return 0.0, 0.0, 0.0, 0.0, 0.0, 0
    dates = pd.to_datetime(tradebook['Entry Time'], errors='coerce')
    pnls = tradebook['PnL'].values
    equity = np.cumsum(pnls)
    total_profit = equity[-1]
    roll_max = np.maximum.accumulate(equity)
    drawdown = roll_max - equity
    max_drawdown = np.max(drawdown)
    ratio = total_profit / max_drawdown if max_drawdown > 0 else float('inf') if total_profit > 0 else 0.0
    x = np.arange(len(equity))
    y = equity
    x_mean, y_mean = x.mean(), y.mean()
    ss_tot = np.sum((y - y_mean)**2)
    slope = np.dot(x - x_mean, y - y_mean) / np.dot(x - x_mean, x - x_mean)
    y_pred = slope * (x - x_mean) + y_mean
    ss_res = np.sum((y - y_pred)**2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0
    months = pd.to_datetime(dates).values.astype('datetime64[M]')
    month_pnls = pd.Series(pnls).groupby(months).sum().values
    monthly_consistency = month_pnls.mean() / month_pnls.std() if len(month_pnls) > 1 and month_pnls.std() > 0 else 0.0
    total_trades = len(pnls)
    win_rate = np.mean(pnls > 0)
    return ratio, total_profit, r2, monthly_consistency, win_rate, total_trades

# === Main Tuning Function ===
def main():
    global data  # use your already-loaded DataFrame
    start_index = 0
    results = []

    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            checkpoint = pickle.load(f)
            start_index = checkpoint['last_index'] + 1
            results = checkpoint['results']
            print(f"🔁 Resuming from checkpoint at index {start_index}")

    SAVE_EVERY = 100
    SAVE_INTERVAL = 300  # seconds
    last_save_time = time.time()

    try:
        for idx in tqdm(range(start_index, len(param_combinations)), desc="Hyperparameter Tuning"):
            combo = param_combinations[idx]
            param = dict(zip(keys, combo))

            try:
                temp_data = data.copy()
                temp_data = temp_data.groupby("Ticker", group_keys=False).apply(
                    lambda df: compute_jma_indicator(
                        df,
                        smooth_length=param['smooth_length'],
                        normalization_period=param['norm_period'],
                        upper_threshold=param['upper_threshold'],
                        lower_threshold=param['lower_threshold'],
                        rs_period=param['rs_period'],
                        rs_percentile=param['rs_percentile']
                    )
                )

                temp_data['Signal'] = temp_data.groupby('Ticker')['Signal'].shift(1)
                temp_data.dropna(subset=['Signal'], inplace=True)
                temp_data['Signal'] = temp_data['Signal'].astype(int)

                tradebook = backtest_trades(
                    temp_data,
                    sl_pct=param['sl_pct'],
                    tp_pct=param['tp_pct'],
                    entry_sl_pct=param['entry_sl_pct']
                )

                metrics = calculate_metrics_fast(tradebook)
                result = {
                    **param,
                    'Profit/Drawdown': metrics[0],
                    'Total Profit': metrics[1],
                    'Equity R2': metrics[2],
                    'Monthly Consistency': metrics[3],
                    'Win Rate': metrics[4],
                    'Total Trades': metrics[5],
                }
                result['Composite Score'] = (
                    result['Profit/Drawdown'] * 0.3 +
                    result['Equity R2'] * 0.5 +
                    result['Monthly Consistency'] * 0.2
                )

                results.append(result)
                print(f"✅ {idx}: {result}")

                if (
                    len(results) % SAVE_EVERY == 0 or
                    time.time() - last_save_time > SAVE_INTERVAL or
                    idx == len(param_combinations) - 1
                ):
                    df = pd.DataFrame(results)
                    df.to_csv(csv_path, index=False)
                    with open(cache_path, 'wb') as f:
                        pickle.dump({'last_index': idx, 'results': results}, f)
                    last_save_time = time.time()
                    print(f"💾 Saved checkpoint at index {idx}")

            except Exception as e:
                print(f"❌ Error on combo {idx}: {param} — {e}")
                continue

    finally:
        df = pd.DataFrame(results)
        df.to_csv(csv_path, index=False)
        with open(cache_path, 'wb') as f:
            pickle.dump({'last_index': idx, 'results': results}, f)
        print(f"💾 Final checkpoint saved at index {idx}")

# === Run Tuning ===
if __name__ == "__main__":
    main()
